<a href="https://colab.research.google.com/github/renisha04/Machine-Learning-Lab/blob/main/ML_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STUDENT PERFORMANCE DATA PREPROCESSING

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


In [5]:
df = pd.read_csv("student_performance_updated_1000.csv")

print("Dataset loaded successfully!")
print("Original Shape:", df.shape)

Dataset loaded successfully!
Original Shape: (1000, 12)


In [18]:
print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Dataset Information ---")
df.info()

print("\n--- Statistical Summary ---")
print(df.describe(include="all"))

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print(df.duplicated().sum())


--- First 5 Rows ---
   gender  attendancerate  studyhoursperweek  previousgrade  \
0       1            85.0               15.0           78.0   
1       0            90.0               20.0           85.0   
2       1            78.0               10.0           65.0   
3       1            92.0               25.0           90.0   
4       0            88.0               18.0           82.0   

   extracurricularactivities  finalgrade  study_hours  attendance_(percent)  \
0                        1.0        80.0          4.8                  59.0   
1                        2.0        87.0          2.2                  70.0   
2                        0.0        68.0          4.6                  92.0   
3                        3.0        92.0          2.9                  96.0   
4                        2.0        85.0          4.1                  97.0   

   online_classes_taken  parentalsupport_Low  parentalsupport_Medium  
0                     0                    0         

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "percent", regex=False)
)

print("\n--- Standardized Columns ---")
print(df.columns.tolist())



--- Standardized Columns ---
['studentid', 'name', 'gender', 'attendancerate', 'studyhoursperweek', 'previousgrade', 'extracurricularactivities', 'parentalsupport', 'finalgrade', 'study_hours', 'attendance_(percent)', 'online_classes_taken']


In [8]:
unnecessary_columns = ["studentid", "name"]

df.drop(
    columns=unnecessary_columns,
    errors="ignore",
    inplace=True
)

print("\nAfter removing unnecessary columns:")
print(df.columns.tolist())



After removing unnecessary columns:
['gender', 'attendancerate', 'studyhoursperweek', 'previousgrade', 'extracurricularactivities', 'parentalsupport', 'finalgrade', 'study_hours', 'attendance_(percent)', 'online_classes_taken']


In [9]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

if "attendancerate" in df.columns:
    df.loc[
        (df["attendancerate"] < 0) |
        (df["attendancerate"] > 100),
        "attendancerate"
    ] = np.nan

if "attendance_percent" in df.columns:
    df.loc[
        (df["attendance_percent"] < 0) |
        (df["attendance_percent"] > 100),
        "attendance_percent"
    ] = np.nan

for column in ["previousgrade", "finalgrade"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) |
            (df[column] > 100),
            column
        ] = np.nan

for column in ["studyhoursperweek", "study_hours"]:
    if column in df.columns:
        df.loc[
            df[column] < 0,
            column
        ] = np.nan

print("\nInvalid values handled.")



Invalid values handled.


In [19]:
before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

print("\nDuplicates removed:", before - after)
print("Shape after removing duplicates:", df.shape)


Duplicates removed: 0
Shape after removing duplicates: (1000, 11)


In [20]:

for column in df.select_dtypes(include=["bool"]).columns:
    df[column] = df[column].astype(int)

categorical_columns = df.select_dtypes(
    include=["object"]
).columns

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("\nEncoding completed.")
print("Shape after encoding:", df.shape)


Encoding completed.
Shape after encoding: (1000, 11)


In [21]:

target_column = "finalgrade"

if target_column not in df.columns:
    raise ValueError(
        "Target column 'finalgrade' was not found. "
        "Check the column name in your dataset."
    )

X = df.drop(columns=[target_column])
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n--- Train-Test Split ---")
print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


--- Train-Test Split ---
Training data: (800, 10)
Testing data : (200, 10)


In [28]:
numeric_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

numeric_imputer = SimpleImputer(strategy="median")

X_train[numeric_columns] = numeric_imputer.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = numeric_imputer.transform(
    X_test[numeric_columns]
)

other_columns = X_train.columns.difference(
    numeric_columns
)

if len(other_columns) > 0:

    categorical_imputer = SimpleImputer(
        strategy="most_frequent"
    )

    X_train[other_columns] = categorical_imputer.fit_transform(
        X_train[other_columns]
    )

    X_test[other_columns] = categorical_imputer.transform(
        X_test[other_columns]
    )

print("\nMissing value imputation completed.")

print("Missing values in training data:")
print(X_train.isnull().sum().sum())

print("Missing values in testing data:")
print(X_test.isnull().sum().sum())



Missing value imputation completed.
Missing values in training data:
0
Missing values in testing data:
0


In [24]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("\nFeature scaling completed.")


Feature scaling completed.


In [27]:

train_cleaned = pd.concat(
    [X_train_scaled, y_train],
    axis=1
)

test_cleaned = pd.concat(
    [X_test_scaled, y_test],
    axis=1
)

train_cleaned.to_csv(
    "student_performance_cleaned_train.csv",
    index=False
)

test_cleaned.to_csv(
    "student_performance_cleaned_test.csv",
    index=False
)

In [29]:

print("\n============================================")
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("============================================")

print("\nFinal Training Shape:",
      train_cleaned.shape)

print("Final Testing Shape:",
      test_cleaned.shape)

print("\nMissing values in training:",
      train_cleaned.isnull().sum().sum())

print("Missing values in testing:",
      test_cleaned.isnull().sum().sum())

print("\nSaved files:")
print("student_performance_cleaned_train.csv")
print("student_performance_cleaned_test.csv")


PREPROCESSING COMPLETED SUCCESSFULLY

Final Training Shape: (800, 11)
Final Testing Shape: (200, 11)

Missing values in training: 0
Missing values in testing: 0

Saved files:
student_performance_cleaned_train.csv
student_performance_cleaned_test.csv
